# Sampler sweep (epoch-300 model)

Tries a clipped DDIM-style sampler (100/250 steps, eta 0/1) against the same 2,000-image protocol, then compares checkpoints.

In [1]:
!pip install -q torch-fidelity diffusers

In [2]:
RUN       = "G_finetune"; ARCH = "pretrained"; SCHEDULE = "linear"
SWEEP_N   = 2000
SAMPLERS  = [dict(steps=100, eta=0.0), dict(steps=100, eta=1.0),
             dict(steps=250, eta=0.0), dict(steps=250, eta=1.0)]
CKPTS     = ["weights_ep100.pt", "weights_ep150.pt", "weights_ep200.pt", "weights_ep250.pt", "model.pt"]
CKPT_SWEEP_STEPS = 100
FINAL_N   = 10000
BS        = 250
SEED      = 0

In [3]:
import os, sys, glob, json, math, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, torch, torchvision, kagglehub
import torchvision.transforms as T
from torchvision.utils import save_image
from tqdm.auto import tqdm
from diffusers import UNet2DModel
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore

assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2"
DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
DEV = DEVS[0]
OUT = "/kaggle/working/sweep"; os.makedirs(OUT, exist_ok=True)
print("GPUs:", [torch.cuda.get_device_name(d) for d in DEVS])

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

assert find(f"{RUN}/ema.pt"), "Attach the G training notebook (Add Input -> Notebooks -> ddpm-task)."
print("G weights dir:", os.path.dirname(find(f"{RUN}/ema.pt")))
print("baseline weights:", find("baseline_asis/model.pt"))

hits = sorted(glob.glob("/kaggle/input/**/flowers-recognition", recursive=True))
path = hits[0] if hits else kagglehub.dataset_download("alxmamaev/flowers-recognition")
tf = T.Compose([T.ToTensor(), T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)])
ds = torchvision.datasets.ImageFolder(root=path, transform=tf)
real_u8 = (torch.stack([ds[i][0] for i in range(len(ds))]).clamp(0, 1) * 255).round().to(torch.uint8)
print("real:", tuple(real_u8.shape))

TSTEPS = 1000
beta = torch.linspace(1e-4, 0.02, TSTEPS, dtype=torch.float32)
a_bar = torch.cumprod(1 - beta, 0)

def build_model(arch):
    if arch == "pretrained":
        return UNet2DModel.from_pretrained("google/ddpm-cifar10-32")
    down = ("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D")
    up   = ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
    return UNet2DModel(in_channels=3, out_channels=3, block_out_channels=[64, 128, 256, 256],
                       layers_per_block=2, down_block_types=down, up_block_types=up)

def load_nets(weights_path, arch):
    sd = torch.load(weights_path, map_location="cpu"); nets = []
    for d in DEVS:
        m = build_model(arch); m.load_state_dict(sd); nets.append(m.to(d).eval())
    return nets

GPUs: ['Tesla T4', 'Tesla T4']
G weights dir: /kaggle/input/notebooks/malharinamdar/ddpm-task/G_finetune
baseline weights: /kaggle/input/notebooks/malharinamdar/notebook11174aea07/baseline_asis/model.pt
real: (4317, 3, 32, 32)


In [4]:
def taus_for(steps):
    return np.linspace(0, TSTEPS - 1, steps).round().astype(int)[::-1].copy()

@torch.no_grad()
def sample_on(i, nets, n, seed, bs, steps, eta, tag=None):
    d, m = DEVS[i], nets[i]; ab = a_bar.to(d); taus = taus_for(steps)
    g = torch.Generator(device=d).manual_seed(seed * 1000 + i); outs = []
    for s in range(0, n, bs):
        b = min(bs, n - s)
        x = torch.randn((b, 3, 32, 32), device=d, generator=g)
        for k, t in enumerate(tqdm(taus, desc=f"gpu{i} {steps}st eta{eta} batch {s//bs+1}/{(n+bs-1)//bs}", position=i, leave=False)):
            a_t = ab[t]
            a_s = ab[taus[k + 1]] if k + 1 < len(taus) else torch.ones((), device=d)
            eps = m(x, torch.full((b,), int(t), dtype=torch.long, device=d)).sample
            x0 = ((x - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
            eps = (x - a_t.sqrt() * x0) / (1 - a_t).sqrt()
            sigma = eta * ((1 - a_s) / (1 - a_t)).sqrt() * (1 - a_t / a_s).sqrt()
            x = a_s.sqrt() * x0 + (1 - a_s - sigma ** 2).clamp(min=0).sqrt() * eps
            if eta > 0 and k + 1 < len(taus):
                x = x + sigma * torch.randn(x.shape, device=d, generator=g)
        outs.append(((x.clamp(-1, 1) + 1) / 2 * 255).round().to(torch.uint8).cpu())
        if tag: np.save(f"{OUT}/{tag}_gpu{i}.npy", torch.cat(outs).numpy())
    return torch.cat(outs)

def sample(nets, n, seed, bs, steps, eta, tag=None):
    k = len(DEVS); shares = [n // k + (1 if i < n % k else 0) for i in range(k)]
    with ThreadPoolExecutor(k) as ex:
        parts = list(ex.map(lambda i: sample_on(i, nets, shares[i], seed, bs, steps, eta, tag), range(k)))
    return torch.cat(parts)

def fid_kid(a_u8, b_u8):
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 500):
            xb = x[i:i + 500].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute()
    return fid.compute().item(), km.item(), ks.item()

sweep = []
def record(row):
    sweep.append(row); print(row)
    json.dump(sweep, open(f"{OUT}/sweep.json", "w"), indent=2)

In [5]:
wdir = os.path.dirname(find(f"{RUN}/ema.pt"))
nets = load_nets(f"{wdir}/ema.pt", ARCH)
for cfg in SAMPLERS:
    t0 = time.time()
    gen = sample(nets, SWEEP_N, SEED, BS, cfg["steps"], cfg["eta"])
    f, km, ks = fid_kid(real_u8, gen)
    record(dict(stage="A_sampler", ckpt="ema.pt", **cfg, fid_2k=f, kid_2k=km, minutes=round((time.time() - t0) / 60, 1)))
best_cfg = min([r for r in sweep if r["stage"] == "A_sampler"], key=lambda r: r["fid_2k"])
print("best sampler:", best_cfg)

{'stage': 'A_sampler', 'ckpt': 'ema.pt', 'steps': 100, 'eta': 0.0, 'fid_2k': 29.20700454711914, 'kid_2k': 0.006964349653571844, 'minutes': 6.0}


{'stage': 'A_sampler', 'ckpt': 'ema.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 28.519033432006836, 'kid_2k': 0.006672568153589964, 'minutes': 6.3}


{'stage': 'A_sampler', 'ckpt': 'ema.pt', 'steps': 250, 'eta': 0.0, 'fid_2k': 29.853336334228516, 'kid_2k': 0.0077102589420974255, 'minutes': 14.1}


{'stage': 'A_sampler', 'ckpt': 'ema.pt', 'steps': 250, 'eta': 1.0, 'fid_2k': 27.289697647094727, 'kid_2k': 0.005971760489046574, 'minutes': 14.1}
best sampler: {'stage': 'A_sampler', 'ckpt': 'ema.pt', 'steps': 250, 'eta': 1.0, 'fid_2k': 27.289697647094727, 'kid_2k': 0.005971760489046574, 'minutes': 14.1}


In [6]:
del nets; torch.cuda.empty_cache()
ref = [r for r in sweep if r["stage"] == "A_sampler" and r["steps"] == CKPT_SWEEP_STEPS and r["eta"] == best_cfg["eta"]]
if ref: record(dict(ref[0], stage="B_ckpt"))
for ck in CKPTS:
    p = f"{wdir}/{ck}"
    if not os.path.exists(p): print("skip (missing):", ck); continue
    t0 = time.time(); nets = load_nets(p, ARCH)
    gen = sample(nets, SWEEP_N, SEED, BS, CKPT_SWEEP_STEPS, best_cfg["eta"])
    f, km, ks = fid_kid(real_u8, gen)
    record(dict(stage="B_ckpt", ckpt=ck, steps=CKPT_SWEEP_STEPS, eta=best_cfg["eta"], fid_2k=f, kid_2k=km,
                minutes=round((time.time() - t0) / 60, 1)))
    del nets; torch.cuda.empty_cache()
best_ck = min([r for r in sweep if r["stage"] == "B_ckpt"], key=lambda r: r["fid_2k"])["ckpt"]
print("best checkpoint:", best_ck, "| best sampler:", best_cfg["steps"], "steps, eta", best_cfg["eta"])

{'stage': 'B_ckpt', 'ckpt': 'ema.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 28.519033432006836, 'kid_2k': 0.006672568153589964, 'minutes': 6.3}


{'stage': 'B_ckpt', 'ckpt': 'weights_ep100.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 32.68971633911133, 'kid_2k': 0.009573759511113167, 'minutes': 6.4}


{'stage': 'B_ckpt', 'ckpt': 'weights_ep150.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 30.212615966796875, 'kid_2k': 0.008169799111783504, 'minutes': 6.4}


> **Note:** this cell ran out of GPU memory in the FID step after three checkpoints (Inception batch too large). The checkpoint comparison and the final evaluation were redone in `04_continue_training_and_final_eval.ipynb`.

In [ ]:
nets = load_nets(f"{wdir}/{best_ck}", ARCH)
t0 = time.time()
gen = sample(nets, FINAL_N, SEED + 1, BS, best_cfg["steps"], best_cfg["eta"], tag="final")
final_minutes = (time.time() - t0) / 60
np.save(f"{OUT}/final_samples.npy", gen.numpy())
save_image(gen[:64].float() / 255, f"{OUT}/final_samples_grid.png", nrow=8)
del nets; torch.cuda.empty_cache()

res = dict(ckpt=best_ck, steps=best_cfg["steps"], eta=best_cfg["eta"], n_total=len(gen), sample_minutes=round(final_minutes, 1))
r2k = [fid_kid(real_u8, gen[j:j + 2000]) for j in range(0, len(gen) - 1999, 2000)]
res["fid_2k"] = [round(r[0], 3) for r in r2k]; res["kid_2k"] = [r[1] for r in r2k]
res["fid_2k_mean"], res["fid_2k_std"] = float(np.mean(res["fid_2k"])), float(np.std(res["fid_2k"]))
res["kid_2k_mean"], res["kid_2k_std"] = float(np.mean(res["kid_2k"])), float(np.std(res["kid_2k"]))
Ns = [n for n in (2000, 4000, 6000, 8000, 10000) if n <= len(gen)]; fid_at = {}
for n in Ns:
    vals = [fid_kid(real_u8, gen[torch.randperm(len(gen), generator=torch.Generator().manual_seed(r))[:n]])[0]
            for r in range(3 if n < len(gen) else 1)]
    fid_at[n] = float(np.mean(vals)); print(f"FID@{n}: {fid_at[n]:.3f}")
res["fid_at_N"] = fid_at
b, a = np.polyfit([1 / n for n in Ns], [fid_at[n] for n in Ns], 1); res["fid_inf"], res["fid_inf_slope"] = float(a), float(b)
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2); print("core final results saved:", res["fid_2k_mean"], fid_at.get(10000), res["fid_inf"])

def inception_score(x_u8):
    m = InceptionScore().to(DEV)
    for i in range(0, len(x_u8), 500): m.update(x_u8[i:i + 500].to(DEV))
    mu, sd = m.compute(); return mu.item(), sd.item()
class U8(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]
def prc(x_u8):
    r = torch_fidelity.calculate_metrics(input1=U8(x_u8[:2000]), input2=U8(real_u8), prc=True, cuda=True, batch_size=250, verbose=False)
    return r["precision"], r["recall"]
def nn_check(x_u8, fname):
    R = real_u8.float().flatten(1) / 255; G = x_u8[:64].float().flatten(1) / 255; exact = "donot_use_mm_for_euclid_dist"
    d_gen, idx = torch.cdist(G, R, compute_mode=exact).min(1)
    q = torch.randperm(len(R), generator=torch.Generator().manual_seed(0))[:64]
    dr = torch.cdist(R[q], R, compute_mode=exact); dr[torch.arange(64), q] = float("inf")
    save_image(torch.stack([x_u8[:64], real_u8[idx]], 1).flatten(0, 1)[:32].float() / 255, f"{OUT}/{fname}", nrow=8)
    return d_gen.mean().item(), dr.min(1).values.mean().item()

try: res["is_mean"], res["is_std"] = inception_score(gen)
except Exception as e: res["is_error"] = repr(e); print("IS failed:", e)
try: res["precision"], res["recall"] = prc(gen)
except Exception as e: res["prc_error"] = repr(e); print("precision/recall failed:", e)
res["nn_dist_gen"], res["nn_dist_real"] = nn_check(gen, "final_nearest_neighbours.png")
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

In [ ]:
def safe(fn):
    try: return fn()
    except Exception as e: print("failed:", e); return repr(e)
base = {}
bw = find("baseline_asis/model.pt"); bs_npy = find("baseline_asis/samples_seed0.npy")
base["is_real"] = safe(lambda: inception_score(real_u8))
if bs_npy:
    g0 = torch.from_numpy(np.load(bs_npy))
    base["orig_sampler"] = dict(fid_kid=safe(lambda: fid_kid(real_u8, g0)), is_=safe(lambda: inception_score(g0)),
                                prc=safe(lambda: prc(g0)))
    json.dump(base, open(f"{OUT}/baseline_results.json", "w"), indent=2)
if bw:
    try:
        nets = load_nets(bw, "orig")
        g1 = sample(nets, SWEEP_N, SEED, BS, best_cfg["steps"], best_cfg["eta"])
        save_image(g1[:64].float() / 255, f"{OUT}/baseline_best_sampler_grid.png", nrow=8)
        del nets; torch.cuda.empty_cache()
        base["best_sampler"] = dict(steps=best_cfg["steps"], eta=best_cfg["eta"], fid_kid=safe(lambda: fid_kid(real_u8, g1)),
                                    is_=safe(lambda: inception_score(g1)), prc=safe(lambda: prc(g1)))
    except Exception as e:
        base["best_sampler_error"] = repr(e); print("baseline resample failed:", e)
json.dump(base, open(f"{OUT}/baseline_results.json", "w"), indent=2)
print(json.dumps(base, indent=2))
print("\nSWEEP SUMMARY"); [print(r) for r in sweep]